# Tutorial 17: Three-Node Mesh on Localhost

Tutorial 9 introduced two-node peering. This tutorial scales to three policies running in the same process, each peered to the other two — a fully connected mesh. Any node can request any other node's data through a peer proxy.

You will:

- Spin up three local policies on three TCP ports
- Pair-peer each combination
- Memorize one entry per node
- Fetch any entry from any node via `laila.universe`
- Disconnect one node and confirm graceful failure for the others

**Prerequisites:** `pip install laila-core`.

In [ ]:
import time
import uuid

import laila
from laila.macros.defaults import DefaultPool, DefaultTCPIPProtocol
from laila.policy.schema.base import _LAILA_IDENTIFIABLE_POLICY

## Helper functions

`_make_policy()` builds a fresh policy with its own TCP listener and an in-memory pool. `_peer(a, b)` connects `a` to `b` using `b`'s bound port and shared secret. These mirror the helpers used in LAILA's communication tests.

In [ ]:
def make_policy(label: str):
    policy = _LAILA_IDENTIFIABLE_POLICY()
    laila._local_policies[policy.global_id] = policy
    laila.active_policy = policy

    tcp = DefaultTCPIPProtocol(host="127.0.0.1", port=0, peer_secret_key=uuid.uuid4().hex)
    policy.central.communication.add_connection(tcp)
    tcp.start()

    pool = DefaultPool()
    policy.central.memory.extend(pool, pool_nickname=f"{label}_store")
    return policy, tcp

def pair(a_policy, a_tcp, b_policy, b_tcp):
    laila.active_policy = a_policy
    a_policy.central.communication.add_tcpip_peer(
        "127.0.0.1", b_tcp.bound_port, b_tcp.peer_secret_key,
    )

## Build three nodes

In [ ]:
A, tcp_A = make_policy("A")
B, tcp_B = make_policy("B")
C, tcp_C = make_policy("C")

print("A gid:", A.global_id)
print("B gid:", B.global_id)
print("C gid:", C.global_id)
print("ports:", tcp_A.bound_port, tcp_B.bound_port, tcp_C.bound_port)

## Pair-peer every combination

In [ ]:
pair(A, tcp_A, B, tcp_B)
pair(A, tcp_A, C, tcp_C)
pair(B, tcp_B, C, tcp_C)
time.sleep(0.4)

for p in (A, B, C):
    laila.active_policy = p
    print(f"{p.global_id[:8]} peers:", [g[:8] for g in laila.peers.keys()])

## Store one entry per node

Each node owns a different entry, all under its own pool nickname.

In [ ]:
laila.active_policy = A
entry_A = laila.constant(data="payload from A", nickname="payload_A")
laila.memorize(entry_A, pool_nickname="A_store").wait()

laila.active_policy = B
entry_B = laila.constant(data="payload from B", nickname="payload_B")
laila.memorize(entry_B, pool_nickname="B_store").wait()

laila.active_policy = C
entry_C = laila.constant(data="payload from C", nickname="payload_C")
laila.memorize(entry_C, pool_nickname="C_store").wait()

print("entries seeded")

## Fetch B's entry from A via active-policy swap

Setting `laila.active_policy` to a peer proxy routes subsequent calls over the wire.

In [ ]:
laila.active_policy = A
proxy_B = laila.peers[B.global_id]
laila.active_policy = proxy_B

result = laila.remember(entry_B.global_id, pool_nickname="B_store", persist=False).wait()
if isinstance(result, list):
    result = result[0]
print("A fetched from B:", result.data)
laila.active_policy = A

## `laila.universe` — lookup by gid without picking a host

`laila.universe` is the union of local and remote policies, keyed by `global_id`. Useful when you have a gid in hand and do not want to think about whether it is local or remote.

In [ ]:
universe = laila.universe
print("known policies:", [g[:8] for g in universe])
print("C policy reachable from A:", C.global_id in universe)

## Disconnect one node — others stay healthy

Stopping `B`'s communication tears down its server thread and drops it from every peer's `peers` map. Subsequent attempts from `A` to reach `B` raise a connection error; `C` is unaffected.

In [ ]:
B.central.communication.stop()
time.sleep(0.3)

laila.active_policy = A
print("A peers after B stops:", [g[:8] for g in laila.peers.keys()])

In [ ]:
laila.active_policy = A
proxy_C = laila.peers[C.global_id]
laila.active_policy = proxy_C
got = laila.remember(entry_C.global_id, pool_nickname="C_store", persist=False).wait()
if isinstance(got, list):
    got = got[0]
print("A still talks to C:", got.data)
laila.active_policy = A

## Tear down

In [ ]:
errors = laila.terminate(wait=True)
print("teardown errors:", errors)
print("local_policies remaining:", len(laila._local_policies))

## Summary

- Three (or more) local policies can be peered in the same process by giving each its own TCP listener.
- `add_tcpip_peer` is symmetric — pair any two nodes and both directions work.
- `laila.peers` lists peers for the **active** policy; `laila.universe` is the cross-policy view.
- Disconnecting one node degrades cleanly: the other peers drop it from their maps without disturbing each other.